# DID_divorce_event_study

> 对应代码：`EconometricsCode/code_in_notes/Chap.15/DID_divorce_event_study.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


读入 Wolfers (2006, AER) 的美国各州离婚数据，研究单边离婚法（unilateral divorce）改革对离婚率的影响。

In [ ]:
%%stata
clear
set more off
use "../datasets/Divorce-Wolfers-AER.dta"


州名 st 是字符串，用 egen group() 生成数值型州编号 state，供面板设定与聚类使用。

In [ ]:
%%stata
** state dummies
egen state=group(st)


保留 1968–1988 年样本，并用 xtset 声明为州×年的面板数据。

In [ ]:
%%stata
** panel setting
keep if year>1967 & year<1989
xtset state year


基准 TWFE 回归：离婚率 div_rate 对改革虚拟变量 unilateral 回归，控制一系列交互项 divx*，reghdfe 吸收州与年份固定效应，按州人口 stpop 加权（[w=stpop]），标准误聚类到州层面。这就是传统的双向固定效应 DID 估计。

In [ ]:
%%stata
** TWFE setting
reghdfe div_rate unilateral divx* [w=stpop], absorb(i.state i.year) cl(state)


构造事件研究的相对时间：never_treated 标记样本期内从未改革的州（lfdivlaw>1989），always_treated 标记样本期之前已改革的州；delta_year 是相对改革年份的时间；L12unilateral 把相对时间 −12 期及以前合并为一个端点虚拟变量。

In [ ]:
%%stata
** prepare relative time
gen never_treated = lfdivlaw>1989
gen always_treated= lfdivlaw<=1967
gen delta_year=year-lfdivlaw
gen L12unilateral = delta_year<=-12


循环生成 L2–L11 的 leads 虚拟变量，分别对应改革前 2 至 11 年（delta_year 取相应的负值）。

In [ ]:
%%stata
forvalues t=11(-1)2{
    gen L`t'unilateral = delta_year==-`t'
}


生成改革当年虚拟变量 unilateral0、改革后 1–19 期的 F1–F19 虚拟变量，并把 +20 期及以后合并为端点 F20unilateral；改革前 1 期（delta_year=−1）被省略，作为事件研究的基期。

In [ ]:
%%stata
gen unilateral0  = delta_year==0
forvalues t=1/19{
    gen F`t'unilateral = delta_year==`t'
}
gen F20unilateral = delta_year>=20


定义处理队列 cohort=lfdivlaw（各州首次改革年份），从未处理州的 cohort 设为缺失；eventstudyweights 计算各相对时间的队列权重（Sun–Abraham 权重），同样吸收州与年份固定效应、控制 divx*，并把权重保存到 weights.xlsx，用于检查每一期系数由哪些处理队列加权而成。

In [ ]:
%%stata
** compute weights
gen cohort=lfdivlaw
replace cohort =. if never_treated
eventstudyweights L12unilateral - F20unilateral [w=stpop], absorb(i.state i.year) cohort(cohort) rel_time(delta_year) covariates(divx*) saveweights("weights.xlsx")


用 eventstudyinteract 实现 Sun & Abraham (2021) 的交互加权估计量：先按“处理队列 × 相对时间”交互估计各期效应，再按样本中各队列的份额加权平均，并以从未处理组为对照（control_cohort(never_treated)）。这样避免了处理效应异质时 TWFE 事件研究中“早处理组充当晚处理组对照”造成的污染。

In [ ]:
%%stata
** estimate
eventstudyinteract div_rate L12unilateral - F20unilateral [w=stpop], absorb(i.state i.year) cohort(cohort) covariates(divx*) control_cohort(never_treated)


绘图：把交互加权系数 e(b_iw) 与方差 e(V_iw) 的对角元开根号（标准误，用 mata 计算）拼入矩阵 C，coefplot 逐期画出系数与置信区间；竖线 xline(12) 对应政策时点（各期系数按时间排序后第 12 个位置即基期所在）。

In [ ]:
%%stata
* plot
matrix C = e(b_iw)
mata st_matrix("A",sqrt(st_matrix("e(V_iw)")))
matrix C = C \ A
coefplot matrix(C[1]), se(C[2])vert xline(12)
